In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt

# 1. 爬第一页
url = 'https://bj.lianjia.com/zufang/'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'lxml')
items = soup.find_all('div', class_='content__list--item')
print(f"抓到 {len(items)} 条房源")

# 2. 提取数据
houses = []
for item in items:
    price_tag = item.find('span', class_='content__list--item-price')
    desc_tag = item.find('p', class_='content__list--item--des')
    houses.append({
        '价格': price_tag.text.strip() if price_tag else '',
        '描述': desc_tag.text.strip().replace('\n', ' ') if desc_tag else ''
    })

df = pd.DataFrame(houses)
df['区'] = df['描述'].str.extract(r'(\S+区)')
df['面积'] = df['描述'].str.extract(r'([\d.]+)㎡')
df['户型'] = df['描述'].str.extract(r'(\d+室\d+厅\d+卫)')
df['价格_元'] = df['价格'].str.extract(r'([\d.]+)').astype(float)
df = df.dropna(subset=['价格_元'])
print(f"清洗后有效: {len(df)} 条")

# 3. 画图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
d_cnt = df['区'].value_counts().head(8)
axes[0].barh(d_cnt.index[::-1], d_cnt.values[::-1], color='#4C72B0')
axes[0].set_title('各区房源数量（链家实时）')
d_price = df.groupby('区')['价格_元'].mean().round(0).sort_values(ascending=False).head(8)
axes[1].barh(d_price.index[::-1], d_price.values[::-1], color='#DD8452')
axes[1].set_title('各区平均租金')
plt.tight_layout()
plt.show()

print(f'北京均价: {df["价格_元"].mean():.0f} 元/月')
print(d_price)

df.to_csv('lianjia_bj_rent.csv', index=False, encoding='gbk')
print('已保存 CSV')


抓到 30 条房源
清洗后有效: 30 条


<Figure size 1400x500 with 2 Axes>

北京均价: 5809 元/月
区
北京经济技术开发区-亦庄河西区    12000.0
海淀区                 7300.0
朝阳区                 6920.0
朝阳区-安贞-安华西里社区       6600.0
海淀区-清河-清缘里中区        6400.0
东城区                 6050.0
石景山区                5850.0
大兴区                 5300.0
Name: 价格_元, dtype: float64
已保存 CSV


In [8]:
# 检查第二页实际返回了什么
url = 'https://bj.lianjia.com/zufang/pg2/'
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'lxml')

# 看看页面上有没有房源列表
items = soup.find_all('div', class_='content__list--item')
print(f"第二页房源数: {len(items)}")
print(f"页面标题: {soup.title.text if soup.title else '无'}")

# 如果是0，看看页面里有什么提示
if len(items) == 0:
    # 可能反爬了，看body内容
    body = soup.find('body')
    print(f"\n页面body前500字符:\n{body.text[:500] if body else '无body'}")


第二页房源数: 0
页面标题: 登录

页面body前500字符:




  window.__INITIAL_STATE__ = {
    frame: {
      viewStyle: {},
      adxDomain: "https:\/\/ex.ke.com\/",
      endpoint: {
        adxDomain: "https:\/\/ex.ke.com\/",
        captchaDomain: "https:\/\/captcha.lianjia.com"
      }
    }
  };
  window.__PUBLIC_PATH__ = "\/\/s1.ljcdn.com\/passport-web\/";







# 链家北京租房 — Python 爬虫实战

## 项目背景
- 使用 Python 爬虫实时获取链家北京租房信息
- 覆盖发请求 → 解析 HTML → 数据清洗 → 可视化 → 存储全流程

## 技术实现
- **Requests + BeautifulSoup**：模拟浏览器请求，解析 HTML 提取房源数据
- **正则表达式**：从文本中提取区域、面积、户型、价格
- **Pandas**：数据清洗与结构化
- **Matplotlib**：可视化各区房源数量与均价

## 爬虫的核心步骤
1. 发送 GET 请求（伪装 User-Agent 不被屏蔽）
2. BeautifulSoup 解析 HTML，find_all 定位房源列表
3. 正则提取价格、面积、区、户型
4. 保存为 CSV

## 数据结果（链家实时）
- 30 条北京租房真实数据
- 北京均价约 **元/月
- 各区价格差异明显

## 项目地址
- CSV 文件：lianjia_bj_rent.csv
